In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import torch
from darts.metrics import err
from darts.models import GlobalNaiveSeasonal

from aare.evaluation.custom_metrics.daily_peak import dpd
from aare.evaluation.evaluation import historical_forecasts
from aare.feature_set import FeatureSet
from aare.features.registry import FEATURES
from aare.params import read_params

# Evaluation metrics for detailed analysis

Extracting the raw errors from a set of historical forecasts to do more detailed analysis on them, e.g. by season.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
torch.set_float32_matmul_precision("medium")

In [ ]:
params = read_params()
tz = params["general"]["timezone"]

In [ ]:
model = GlobalNaiveSeasonal(input_chunk_length=24, output_chunk_length=1)

In [ ]:
ds = FeatureSet(targets=FEATURES["temp_bern"], split_params=params["split"])
val = ds.get_val()[0]
len(val[0])

In [ ]:
stride = 1
horizon = 96
hf = historical_forecasts(model, val, horizon, stride)
len(hf[0])

In [ ]:
bt = model.backtest(
    val,
    historical_forecasts=hf,
    metric=[err, dpd],
    metric_kwargs=[{}, dict(tz=tz)],
    reduction=None,
)
bt

In [ ]:
res_err = model.residuals(
    val,
    historical_forecasts=hf,
    metric=err,
    last_points_only=False,
)
res_err

In [ ]:
res_dpd = model.residuals(
    val,
    historical_forecasts=hf,
    metric=dpd,
    last_points_only=False,
    metric_kwargs=dict(tz=tz),
)
res_dpd

In [ ]:
np.concatenate(bt, axis=0).shape

In [ ]:
x = np.concatenate(bt, axis=0)
x = x.reshape(-1, x.shape[-1])
x.shape

In [ ]:
times = np.stack([fc.time_index.values for fc_l in hf for fc in fc_l])
times.shape

In [ ]:
run_ts = times[:, 0] - pd.Timedelta(1, "s")
run_ts.shape

In [ ]:
run_ts.repeat(times.shape[1]).reshape(-1, times.shape[1])

In [ ]:
np.expand_dims(np.stack([fc.time_index.values for fc_l in hf for fc in fc_l]), 2).shape

In [ ]:
df = pd.DataFrame(
    x, index=np.stack([fc.time_index.values for fc_l in hf for fc in fc_l]).ravel(), columns=["err", "dpd"]
)
df

In [ ]:
df["run_ts"] = run_ts.repeat(times.shape[1])
df

In [ ]:
df = df.reset_index(names="time")[["run_ts", "time", "err", "dpd"]]
# df.to_csv("test.csv", index=False)
df

In [ ]:
df["age"] = df["time"] - df["run_ts"]
df

In [ ]:
df["ae"] = df["err"].abs()
df["adpd"] = df["dpd"].abs()

In [ ]:
df.drop(["run_ts", "time"], axis="columns").groupby("age").mean()

In [ ]:
df.melt("age", ["err", "dpd", "ae", "adpd"], var_name="metric")

In [ ]:
px.violin(
    df.melt("age", ["err", "dpd", "ae", "adpd"], var_name="metric"), x="age", y="value", facet_row="metric", box=True
)

## Notes/TODOs

- For anything that does complex data manipulation and analytics, and isn't tied to darts, use polars, it might be worth it. But tbf you need proficiency in both anyway.
- The code that generates the report for the model should take as input something that can be constructed easily from the timescaledb database. E.g. a polars lazyframe with run_ts, time, err and dpd.
- If you want to analyze the residuals with regard to actual values, e.g. does the model overestimate temperature when the sun isn't shining?, you'll need to join with other data based on time (asof). That should also fall under the same logic that works without darts and is a step after calculating the residuals (caveat: when evaluating the model you already have the validation data in memory, reusing that somehow (transforming darts to polars instead of influx to polars) is probably worth it. The join can be agnostic though.
- Sidestepping darts metrics further is probably not worth it because the integration with automatic logging during training of torch models is really nice. You could do both; keep darts metric during training but eval after training would be done by transforming historical forecasts into polars or whatever and doing the calculation there; would 100% be faster. Then you need 3 DPD impls though, darts metric, polars aggregation and SQL aggregation.
- You'll probably have to re-implement DPD in SQL for real-time reporting in Grafana, but I think that's fine (just watch out for timezone stuff).
- Using historical forecasts once and then backtest once for the aggregated metrics and once for the point metrics works well. Using the residual function does not work because that can only take one metric.
- DPD implementation is really slow, depending on how fast models are to train, evaluation could become the bottleneck (for LR it's definitely the case).